# PVI QRF validation: calibration, loss, and skill

This notebook reads the full five-fold out-of-fold validation summaries and creates a compact three-panel publication figure.

- **(a) q90 coverage:** annual empirical coverage relative to the nominal 0.90 level.
- **(b) Pinball loss:** annual QRF q90 loss compared with the empirical-q90 baseline.
- **(c) Pinball skill:** annual relative reduction in pinball loss compared with the baseline.

The figure evaluates statistical calibration and spatial-transfer performance of the conditional quantile. It does not treat PVI as directly observed vegetation truth.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(r"E:\MPLD\PVI\PVI_QRF_V3_q90")
RESULTS = ROOT / "05_validation" / "qrf_final_validation" / "focused_results"
OUT = ROOT / "10_figures" / "publication"
OUT.mkdir(parents=True, exist_ok=True)

ANNUAL_FILE = RESULTS / "annual_metrics.csv"
OVERALL_FILE = RESULTS / "overall_metrics.csv"

annual_all = pd.read_csv(ANNUAL_FILE)
overall_all = pd.read_csv(OVERALL_FILE)

annual = annual_all.loc[annual_all["protocol"].eq("full_5fold_oof")].copy()
overall = overall_all.loc[overall_all["protocol"].eq("full_5fold_oof")].copy()

required = {
    "year", "q90_coverage", "q90_pinball", "baseline_q90_pinball",
    "pinball_relative_improvement",
}
missing = required.difference(annual.columns)
if missing:
    raise ValueError(f"Missing required annual columns: {sorted(missing)}")
if len(overall) != 1:
    raise ValueError(f"Expected one full_5fold_oof overall row; found {len(overall)}")

annual["year"] = annual["year"].astype(int)
annual = annual.sort_values("year").reset_index(drop=True)
expected_years = list(range(2000, 2025))
if annual["year"].tolist() != expected_years:
    raise ValueError("Annual validation rows do not cover each year from 2000 to 2024 exactly once.")
if annual[list(required - {"year"})].isna().any().any():
    raise ValueError("Missing values found in required annual metrics.")
if (annual["baseline_q90_pinball"] <= 0).any():
    raise ValueError("Baseline pinball loss must be positive.")

# Recompute skill from the two losses and verify the supplied summary column.
annual["pinball_skill_percent"] = (
    1.0 - annual["q90_pinball"] / annual["baseline_q90_pinball"]
) * 100.0
reported_skill = annual["pinball_relative_improvement"] * 100.0
if not np.allclose(annual["pinball_skill_percent"], reported_skill, atol=1e-8):
    raise ValueError("Recomputed annual pinball skill differs from the source summary.")

o = overall.iloc[0]
overall_coverage = float(o["q90_coverage"])
overall_loss = float(o["q90_pinball"])
overall_baseline_loss = float(o["baseline_q90_pinball"])
overall_skill = float(o["pinball_relative_improvement"]) * 100.0

source_data = annual[[
    "year", "n", "q90_coverage", "q90_calibration_error",
    "q90_pinball", "baseline_q90_pinball", "pinball_skill_percent",
]].copy()
source_data.to_csv(OUT / "Fig_PVI_q90_validation_metrics_source_data.csv", index=False)

summary = pd.DataFrame({
    "metric": ["q90 empirical coverage", "q90 pinball loss",
               "empirical-baseline pinball loss", "pinball skill (%)"],
    "value": [overall_coverage, overall_loss, overall_baseline_loss, overall_skill],
})
summary.to_csv(OUT / "Fig_PVI_q90_validation_metrics_summary.csv", index=False)

display(summary)

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.5,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

blue = "#2F6F9F"
orange = "#D9772B"
dark = "#333333"
mid_gray = "#9C9C9C"

def four_spines(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_color(dark)
    ax.tick_params(direction="out", colors=dark, pad=2)

# ==================== 1. Figure geometry: mm ====================
LEFT = 12.0
RIGHT = 4.0
BOTTOM = 5.0
TOP = 3.0
AX_W = 50.0
AX_H = 28.0
VGAP = 3.0

FIG_W = LEFT + AX_W + RIGHT
FIG_H = BOTTOM + AX_H * 3 + VGAP * 2 + TOP

YLABEL_X = -0.16
PANEL_X = 0.02
PANEL_Y = 0.98

def axes_rect_mm(left_mm, bottom_mm, width_mm, height_mm):
    return [left_mm / FIG_W, bottom_mm / FIG_H, width_mm / FIG_W, height_mm / FIG_H]

# ==================== 2. Data ====================
years = annual["year"].to_numpy()
tick_years = [2000, 2006, 2012, 2018, 2024]

# Read label values from the supplied validation summary.
overall_qrf_text = overall_loss
baseline_text = overall_baseline_loss

# ==================== 3. Create figure ====================
fig = plt.figure(figsize=(FIG_W / 25.4, FIG_H / 25.4), constrained_layout=False)

ax_a = fig.add_axes(axes_rect_mm(LEFT, BOTTOM + 2 * (AX_H + VGAP), AX_W, AX_H))
ax_b = fig.add_axes(axes_rect_mm(LEFT, BOTTOM + AX_H + VGAP, AX_W, AX_H))
ax_c = fig.add_axes(axes_rect_mm(LEFT, BOTTOM, AX_W, AX_H))
axes = [ax_a, ax_b, ax_c]

# ==================== 4. Panel (a) ====================
ax = ax_a
ax.plot(years, annual["q90_coverage"], color=blue, lw=1.35, marker="o", ms=2.6, mec="white", mew=0.35)
ax.axhline(0.90, color=orange, lw=1.0, ls=(0, (4, 2)))
ax.set_xlim(1999.2, 2024.8)
ax.set_ylim(0.855, 0.925)
ax.set_xticks(tick_years)
ax.set_yticks([0.86, 0.88, 0.90, 0.92])
ax.set_ylabel("Empirical coverage")
ax.text(PANEL_X, PANEL_Y, "(a)", transform=ax.transAxes, ha="left", va="top", fontsize=9)
ax.text(0.04, 0.08, f"Overall = {overall_coverage:.3f}\nNominal = 0.900",
        transform=ax.transAxes, va="bottom", ha="left", fontsize=7.0)
ax.tick_params(labelbottom=False)
four_spines(ax)

# ==================== 5. Panel (b) ====================
ax = ax_b
baseline_series = annual["baseline_q90_pinball"].to_numpy()
qrf_series = annual["q90_pinball"].to_numpy()

ax.plot(years, baseline_series, color=mid_gray, lw=1.15, marker="o", ms=2.3)
ax.plot(years, qrf_series, color=blue, lw=1.45, marker="o", ms=2.6)
ax.set_xlim(1999.2, 2024.8)
ax.set_ylim(0.006, 0.0205)
ax.set_xticks(tick_years)
ax.set_yticks([0.008, 0.012, 0.016, 0.020])
ax.set_ylabel("Pinball loss")
ax.text(PANEL_X, PANEL_Y, "(b)", transform=ax.transAxes, ha="left", va="top", fontsize=9)

# 灰色线标签：放在线下方
x_base = 2004.0
y_base = np.interp(x_base, years, baseline_series)
ax.text(x_base, y_base - 0.0009, f"Baseline = {baseline_text:.4f}",
        color=mid_gray, fontsize=6.8, ha="left", va="top")

# 蓝色线标签：放在线上方
x_qrf = 2008.0
y_qrf = np.interp(x_qrf, years, qrf_series)
ax.text(x_qrf, y_qrf + 0.0009, f"Overall QRF = {overall_qrf_text:.4f}",
        color=blue, fontsize=6.8, ha="left", va="bottom")

ax.tick_params(labelbottom=False)
four_spines(ax)

# ==================== 6. Panel (c) ====================
ax = ax_c
skill = annual["pinball_skill_percent"].to_numpy()
retained_loss = 100.0 - skill
ax.bar(years, retained_loss, width=0.72, color=blue, edgecolor="none", label="QRF loss retained")
ax.bar(years, skill, bottom=retained_loss, width=0.72, color=orange, alpha=0.82, edgecolor="none", label="Loss reduction")
ax.axhline(100, color=dark, lw=0.75, ls=(0, (3, 2)))
ax.set_xlim(1999.2, 2024.8)
ax.set_ylim(0, 104)
ax.set_xticks(tick_years)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_ylabel("Share of baseline loss (%)")
ax.text(PANEL_X, PANEL_Y, "(c)", transform=ax.transAxes, ha="left", va="top", fontsize=9)

leg = ax.legend(loc="lower left", fontsize=6.2, handlelength=1.2, borderaxespad=0.25, labelspacing=0.15, frameon=True)
leg.get_frame().set_facecolor("white")
leg.get_frame().set_edgecolor("none")
leg.get_frame().set_alpha(0.86)

# 右上角标签往下一点
ax.text(0.96, 0.84, f"Overall reduction = {overall_skill:.1f}%",
        transform=ax.transAxes, va="top", ha="right", fontsize=6.8,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "boxstyle": "round,pad=0.18"})
four_spines(ax)

# ==================== 7. Unified alignment ====================
for ax in axes:
    ax.set_xlabel("")
    ax.yaxis.set_label_coords(YLABEL_X, 0.5)

# ==================== 8. Check exact physical dimensions ====================
fig.canvas.draw()
for name, ax in zip(["a", "b", "c"], axes):
    bbox = ax.get_position()
    print(f"Panel ({name}): left={bbox.x0 * FIG_W:.2f} mm, bottom={bbox.y0 * FIG_H:.2f} mm, width={bbox.width * FIG_W:.2f} mm, height={bbox.height * FIG_H:.2f} mm")

print(f"Figure size = {FIG_W:.2f} × {FIG_H:.2f} mm")
print(f"Panel size = {AX_W:.2f} × {AX_H:.2f} mm")
print(f"Vertical gap = {VGAP:.2f} mm")

# ==================== 9. Save ====================
stem = OUT / "Fig_PVI_q90_validation_metrics_vertical_compact"
fig.savefig(stem.with_suffix(".svg"), facecolor="white", bbox_inches=None)
fig.savefig(stem.with_suffix(".pdf"), facecolor="white", bbox_inches=None)
fig.savefig(stem.with_suffix(".png"), dpi=600, facecolor="white", bbox_inches=None)
fig.savefig(stem.with_suffix(".tif"), dpi=600, facecolor="white", bbox_inches=None,
            pil_kwargs={"compression": "tiff_lzw"})

plt.show()

In [ ]:
# EXPORT_SEPARATE_PVI_VALIDATION_PANELS
# Export each panel at a standard single-column width (89 mm).

def save_single_panel(fig, filename):
    stem = OUT / filename
    fig.savefig(stem.with_suffix(".svg"), facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), facecolor="white")
    fig.savefig(stem.with_suffix(".png"), dpi=600, facecolor="white")
    fig.savefig(stem.with_suffix(".tif"), dpi=600, facecolor="white",
                pil_kwargs={"compression": "tiff_lzw"})


def new_single_panel():
    fig, ax = plt.subplots(figsize=(89 / 25.4, 68 / 25.4))
    fig.subplots_adjust(left=0.185, right=0.975, bottom=0.185, top=0.910)
    return fig, ax


# (a) q90 coverage
fig_a, ax = new_single_panel()
ax.plot(years, annual["q90_coverage"], color=blue, lw=1.35,
        marker="o", ms=2.8, mec="white", mew=0.35)
ax.axhline(0.90, color=orange, lw=1.0, ls=(0, (4, 2)))
ax.set_xlim(1999.2, 2024.8)
ax.set_ylim(0.855, 0.925)
ax.set_xticks(tick_years)
ax.set_yticks([0.86, 0.88, 0.90, 0.92])
ax.set_xlabel("Year")
ax.set_ylabel("Empirical coverage")
ax.set_title("(a) q90 coverage", loc="left", fontsize=7.8,
             fontweight="normal", pad=4)
ax.text(0.04, 0.08,
        f"Overall = {overall_coverage:.3f}\nNominal = 0.900",
        transform=ax.transAxes, va="bottom", ha="left", fontsize=6.6)
four_spines(ax)
save_single_panel(fig_a, "Fig_PVI_q90_validation_a_coverage")
plt.show()


# (b) q90 pinball loss
fig_b, ax = new_single_panel()
ax.plot(years, annual["baseline_q90_pinball"], color=mid_gray,
        lw=1.15, marker="o", ms=2.4, label="Empirical q90 baseline")
ax.plot(years, annual["q90_pinball"], color=blue,
        lw=1.45, marker="o", ms=2.7, label="QRF q90")
ax.set_xlim(1999.2, 2024.8)
ax.set_ylim(0.006, 0.0205)
ax.set_xticks(tick_years)
ax.set_yticks([0.008, 0.012, 0.016, 0.020])
ax.set_xlabel("Year")
ax.set_ylabel("Pinball loss")
ax.set_title("(b) q90 pinball loss", loc="left", fontsize=7.8,
             fontweight="normal", pad=4)
ax.legend(loc="upper left", fontsize=6.2, handlelength=2.0,
          borderaxespad=0.25, labelspacing=0.25)
ax.text(0.96, 0.07,
        f"Overall QRF = {overall_loss:.4f}\nBaseline = {overall_baseline_loss:.4f}",
        transform=ax.transAxes, va="bottom", ha="right", fontsize=6.4)
four_spines(ax)
save_single_panel(fig_b, "Fig_PVI_q90_validation_b_pinball_loss")
plt.show()


# (c) relative loss reduction
fig_c, ax = new_single_panel()
skill = annual["pinball_skill_percent"].to_numpy()
retained_loss = 100.0 - skill
ax.bar(years, retained_loss, width=0.72, color=blue, edgecolor="none",
       label="QRF loss retained")
ax.bar(years, skill, bottom=retained_loss, width=0.72,
       color=orange, alpha=0.82, edgecolor="none",
       label="Loss reduction (skill)")
ax.axhline(100, color=dark, lw=0.75, ls=(0, (3, 2)))
ax.set_xlim(1999.2, 2024.8)
ax.set_ylim(0, 104)
ax.set_xticks(tick_years)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_xlabel("Year")
ax.set_ylabel("Share of baseline loss (%)")
ax.set_title("(c) Relative loss reduction", loc="left", fontsize=7.8,
             fontweight="normal", pad=4)
legend = ax.legend(loc="lower left", fontsize=6.0, handlelength=1.4,
                   borderaxespad=0.25, labelspacing=0.25, frameon=True)
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("none")
legend.get_frame().set_alpha(0.86)
ax.text(0.96, 0.94, f"Overall reduction = {overall_skill:.1f}%",
        transform=ax.transAxes, va="top", ha="right", fontsize=6.5,
        color=dark,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82,
              "boxstyle": "round,pad=0.18"})
four_spines(ax)
save_single_panel(fig_c, "Fig_PVI_q90_validation_c_loss_reduction")
plt.show()